In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'MS Gothic'  # Windows\u65e5\u672c\u8a9e\u30d5\u30a9\u30f3\u30c8
import seaborn as sns
from pathlib import Path
import sys
from IPython.display import display

PROJECT_ROOT = Path('.').resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))
from config import settings

TAGS = ['tokyo', 'osaka', 'nagoya', 'fukuoka']
TAG_LABELS = {'tokyo': '\u6771\u4eac', 'osaka': '\u5927\u962a', 'nagoya': '\u540d\u53e4\u5c4b', 'fukuoka': '\u798f\u5ca1'}

# \u7d71\u5408\u30c7\u30fc\u30bf\u8aad\u307f\u8fbc\u307f
dfs = {}
for tag in TAGS:
    path = settings.PROCESSED_DATA_DIR / f'{tag}_integrated.csv'
    if path.exists():
        dfs[tag] = pd.read_csv(path)
        dfs[tag]['area'] = tag
        print(f"{TAG_LABELS[tag]}: {len(dfs[tag])} rows")

all_df = pd.concat(dfs.values(), ignore_index=True)
print(f'\\n\u5408\u8a08: {len(all_df)} rows')

# ML\u30ae\u30e3\u30c3\u30d7\u30c7\u30fc\u30bf
ml_gap_path = settings.PROCESSED_DATA_DIR / 'combined_ml_gap.csv'
ml_gap_df = pd.read_csv(ml_gap_path) if ml_gap_path.exists() else None
if ml_gap_df is not None:
    print(f'ML\u30ae\u30e3\u30c3\u30d7: {len(ml_gap_df)} rows')


## 1. \u30a8\u30ea\u30a2\u57fa\u672c\u7d71\u8a08

In [ ]:
summary = all_df.groupby('area').agg(\n    \u30e1\u30c3\u30b7\u30e5\u6570=('jis_mesh', 'nunique'),\n    \u5e97\u8217\u6570=('restaurant_count', 'sum'),\n    \u5e73\u5747\u4eba\u53e3=('population', 'mean'),\n    \u4e2d\u592e\u5024\u4eba\u53e3=('population', 'median'),\n    \u5e73\u5747\u5e97\u8217\u6570=('restaurant_count', 'mean'),\n    \u30b8\u30e3\u30f3\u30eb\u6570=('unified_genre', 'nunique'),\n).round(1)\nsummary.index = summary.index.map(TAG_LABELS)\nsummary\n

## 2. \u4eba\u53e3\u5206\u5e03\u306e\u6bd4\u8f03

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)\nfor ax, tag in zip(axes, TAGS):\n    df = dfs[tag]\n    mesh_pop = df.groupby('jis_mesh')['population'].first()\n    ax.hist(mesh_pop[mesh_pop > 0], bins=30, alpha=0.7, color='steelblue')\n    ax.set_title(TAG_LABELS[tag])\n    ax.set_xlabel('\u4eba\u53e3')\n    if ax == axes[0]:\n        ax.set_ylabel('\u30e1\u30c3\u30b7\u30e5\u6570')\nplt.suptitle('250m\u30e1\u30c3\u30b7\u30e5\u5358\u4f4d\u306e\u4eba\u53e3\u5206\u5e03', fontsize=14)\nplt.tight_layout()\nplt.show()\n

In [ ]:
mesh_pop = all_df.groupby(['area', 'jis_mesh'])['population'].first().reset_index()\nmesh_pop = mesh_pop[mesh_pop['population'] > 0]\nmesh_pop['area_label'] = mesh_pop['area'].map(TAG_LABELS)\n\nfig, ax = plt.subplots(figsize=(8, 5))\nsns.violinplot(data=mesh_pop, x='area_label', y='population', ax=ax, inner='quartile')\nax.set_title('\u30a8\u30ea\u30a2\u5225 \u4eba\u53e3\u5206\u5e03\uff08250m\u30e1\u30c3\u30b7\u30e5\uff09')\nax.set_xlabel('')\nax.set_ylabel('\u4eba\u53e3')\nplt.tight_layout()\nplt.show()\n

## 3. \u98fd\u548c\u5ea6\uff08saturation_index\uff09\u306e\u6bd4\u8f03

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))\nsat_data = all_df[['area', 'saturation_index']].copy()\nsat_data['area_label'] = sat_data['area'].map(TAG_LABELS)\nsns.boxplot(data=sat_data, x='area_label', y='saturation_index', ax=ax, showfliers=False)\nax.set_title('\u30a8\u30ea\u30a2\u5225 \u98fd\u548c\u5ea6\u5206\u5e03')\nax.set_xlabel('')\nax.set_ylabel('saturation_index')\nplt.tight_layout()\nplt.show()\n

## 4. \u99c5\u8ddd\u96e2\u5206\u5e03\u306e\u6bd4\u8f03

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))\ncolors = ['steelblue', 'coral', 'seagreen', 'mediumpurple']\nfor tag, color in zip(TAGS, colors):\n    if 'nearest_station_distance' in dfs[tag].columns:\n        dist = dfs[tag]['nearest_station_distance']\n        dist = dist[dist > 0]\n        ax.hist(dist, bins=50, alpha=0.5, label=TAG_LABELS[tag], color=color, range=(0, 3))\nax.set_title('\u6700\u5bc4\u308a\u99c5\u8ddd\u96e2\u306e\u5206\u5e03')\nax.set_xlabel('\u8ddd\u96e2 (km)')\nax.set_ylabel('\u884c\u6570')\nax.legend()\nplt.tight_layout()\nplt.show()\n

## 5. \u30b8\u30e3\u30f3\u30eb\u5225\u5e97\u8217\u6570\u306e\u6bd4\u8f03

In [ ]:
genre_area = all_df.groupby(['area', 'unified_genre'])['restaurant_count'].sum().unstack(fill_value=0)\ngenre_area.index = genre_area.index.map(TAG_LABELS)\n\nfig, ax = plt.subplots(figsize=(12, 5))\nsns.heatmap(genre_area, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax)\nax.set_title('\u30b8\u30e3\u30f3\u30eb \u00d7 \u30a8\u30ea\u30a2 \u5e97\u8217\u6570')\nax.set_ylabel('')\nplt.tight_layout()\nplt.show()\n

## 6. ML\u30e2\u30c7\u30eb\u7279\u5fb4\u91cf\u91cd\u8981\u5ea6\u306e\u30a8\u30ea\u30a2\u9593\u6bd4\u8f03

In [ ]:
from src.analyze.ml_model import train_cv\n\narea_importances = {}\narea_metrics = {}\nfor tag in TAGS:\n    if tag not in dfs:\n        continue\n    cv = train_cv(dfs[tag], n_splits=3, target_mode='residual')\n    area_importances[tag] = cv['feature_importance'].set_index('feature')['importance']\n    area_metrics[tag] = {'R2': cv['avg_r2'], 'RMSE': cv['avg_rmse']}\n    print(f"{TAG_LABELS[tag]}: R2={cv['avg_r2']:.4f}, RMSE={cv['avg_rmse']:.4f}")\n\nmetrics_df = pd.DataFrame(area_metrics).T\nmetrics_df.index = metrics_df.index.map(TAG_LABELS)\nprint('\\n')\ndisplay(metrics_df.round(4))\n

In [ ]:
imp_df = pd.DataFrame(area_importances)\nimp_df.columns = [TAG_LABELS[t] for t in imp_df.columns]\n# \u5404\u30a8\u30ea\u30a2\u5185\u3067\u6b63\u898f\u5316\nimp_norm = imp_df.div(imp_df.sum(axis=0), axis=1)\n\nfig, ax = plt.subplots(figsize=(10, 8))\nsns.heatmap(imp_norm, annot=True, fmt='.3f', cmap='Blues', ax=ax)\nax.set_title('\u30a8\u30ea\u30a2\u5225 \u7279\u5fb4\u91cf\u91cd\u8981\u5ea6\uff08\u6b63\u898f\u5316\uff09')\nax.set_ylabel('\u7279\u5fb4\u91cf')\nplt.tight_layout()\nplt.show()\n

## 7. SHAP Summary Plot

In [ ]:
from src.analyze.ml_model import train_full_model, compute_shap_values\n\nfull_model = train_full_model(all_df, target_mode='residual')\nshap_values, features = compute_shap_values(full_model, all_df)\n\nimport shap\nshap.summary_plot(shap_values, features, show=True, plot_type='bar')\n

## 8. \u307e\u3068\u3081

\u5206\u6790\u304b\u3089\u5f97\u3089\u308c\u305f\u77e5\u898b\u3092\u307e\u3068\u3081\u308b\u30d7\u30ec\u30fc\u30b9\u30db\u30eb\u30c0\u30fc\u3002